# 03 - Controlled Clustering Experiments & Model Validation

**Project:** Naval Propulsion Clustering  
**Academic Title:** *Unsupervised Discovery of Operating and Performance Degradation Profiles in Naval Gas Turbine Propulsion Systems*  
**Phase:** Phase 3 — Controlled Clustering Experiments & Model Validation  

This notebook loads and analyzes the empirical clustering experiment outputs produced by `naval_propulsion`.
It addresses our core research questions: operating regime recovery, within-regime degradation isolation, multi-seed stability, algorithm comparisons, and post-hoc statistical validation.

In [ ]:
import json
import matplotlib
matplotlib.use('Agg')  # Non-blocking headless backend
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from naval_propulsion.utils.paths import get_experiments_dir, get_figures_dir
from naval_propulsion.visualization.plots import configure_plot_style

configure_plot_style()

summary_path = get_experiments_dir() / "outputs" / "phase3" / "phase3_summary.json"
with open(summary_path, "r", encoding="utf-8") as f:
    results = json.load(f)
print("Loaded Phase 3 summary results successfully from:", summary_path)

## 1. Operating Regime Recovery: R1 vs R2 vs R3 vs R4 (k=9)

We evaluate whether unconditioned clustering on raw/globally standardized telemetry simply recovers the 9 commanded operating speeds.
Notice that even in Representation R2 (where `v` and `lp` are strictly removed), thermodynamic coupling reconstructs the speed setpoints with ARI > 0.84.

In [ ]:
recovery_rows = []
for rep_id, sweep in results["representation_sweeps"].items():
    k9 = next(item for item in sweep if item["k"] == 9)
    recovery_rows.append({
        "Representation": rep_id,
        "Silhouette (k=9)": k9["silhouette"],
        "Davies-Bouldin": k9["davies_bouldin"],
        "Calinski-Harabasz": k9["calinski_harabasz"],
        "ARI vs Speed": k9["ari_vs_speed"],
        "NMI vs Speed": k9["nmi_vs_speed"],
    })

recovery_df = pd.DataFrame(recovery_rows)
recovery_df

## 2. Primary Degradation Experiment: R5 (Within-Speed Normalized)

Within-speed normalization decouples the 99.33% operating speed variance.
Notice that ARI vs Speed drops to 0.0002 (complete decoupling).

In [ ]:
r5_sweep = results["representation_sweeps"]["R5_WITHIN_SPEED_NORMALIZED"]
r5_df = pd.DataFrame(r5_sweep)
print("R5 K-Means Metric Sweep (k=2 to 12):")
r5_df[["k", "silhouette", "davies_bouldin", "calinski_harabasz", "ari_vs_speed", "nmi_vs_speed"]]

## 3. Post-Hoc Degradation Validation on R5 (Zero-Leakage Analysis)

After unsupervised clustering, we test whether discovered clusters exhibit statistically significant and substantial effect sizes on the quarantined ground truth: `kMc` (Compressor decay) and `kMt` (Turbine decay).

In [ ]:
deg_val = results["degradation_validation"]
deg_rows = []
for model_name, data in deg_val.items():
    deg_rows.append({
        "Model": model_name,
        "kMc eta^2": data["kMc"]["eta_squared"],
        "kMc Kruskal p": data["kMc"]["kruskal_p_value"],
        "kMc Effect Size": data["kMc"]["effect_size_interpretation"],
        "kMt eta^2": data["kMt"]["eta_squared"],
        "kMt Kruskal p": data["kMt"]["kruskal_p_value"],
        "kMt Effect Size": data["kMt"]["effect_size_interpretation"],
    })
pd.DataFrame(deg_rows)

## 4. Multi-Seed Stability Analysis (10 Seeds: 0, 1, 2, 3, 4, 5, 10, 21, 42, 100)

We evaluate pairwise Adjusted Rand Index (45 comparisons) across random seeds.

In [ ]:
stability_df = pd.DataFrame([
    {
        "Model": k,
        "ARI Mean": v["ari_mean"],
        "ARI Std": v["ari_std"],
        "ARI Min": v["ari_min"],
        "ARI Max": v["ari_max"],
        "Stability Rating": v["stability_rating"],
    }
    for k, v in results["stability_analysis"].items()
])
stability_df

## 5. Algorithm Comparison on R5 Space (k=2)

Comparing K-Means, Agglomerative (Ward & Average), and Gaussian Mixture Models.

In [ ]:
algo_df = pd.DataFrame([
    {
        "Algorithm": k,
        "Silhouette": v.get("silhouette", 0.0),
        "Davies-Bouldin": v.get("davies_bouldin", 0.0),
        "Calinski-Harabasz": v.get("calinski_harabasz", 0.0),
    }
    for k, v in results["algorithm_comparison"].items()
])
algo_df

## 6. Generated Publication Figures Overview

10 high-resolution publication figures generated under `reports/figures/phase3/`.

In [ ]:
fig_dir = get_figures_dir() / "phase3"
for f in sorted(fig_dir.glob("*.png")):
    print(f" - {f.name:45s} ({f.stat().st_size:,} bytes)")